In [ ]:
%cd ../../

In [ ]:
import polars as pl
import yaml

from src import Data

# Load data

## Load data dimensions and POS

In [ ]:
data = Data()

### Process meals' info

In [ ]:
meals = (
    data.dim_meals
    .filter(pl.col('restaurant').is_not_null())

    .select('meal_id', 'meal_type', 'restaurant')
    .explode('restaurant')
)

meals.head()

### Process POS

In [ ]:
CUTOFF_DATE = pl.lit("2024-09-01", dtype=pl.Date)

meals_pos = (
    data.pos
    
    # Keep entries in train split
    .filter(pl.col('date') < CUTOFF_DATE)

    
    # Get representative POS for each meal
    .group_by('meal_id', 'meal_type', 'restaurant')
    .agg(pl.col('pcs').median())
)


meals_pos.head()

### Process embeddings

In [ ]:
embeddings = (
    data.dim_embds
    .select(
        'meal_id',
        pl.col('embedding').list.to_array(1024)
    )
)
embeddings.head()

### Combine POS into meal info

In [ ]:
meals = (
    meals

    # Add POS data
    .join(meals_pos, on=['meal_id', 'meal_type', 'restaurant'], how='full')
    .filter(pl.col('meal_id').is_not_null())
    .select(
        'meal_id', 'meal_type', 'restaurant',
        pl.col('pcs').fill_null(-1)
    )

    # Add embedding info
    .join(embeddings, on='meal_id', how='left')
)


# Encode meal_id
meals_ids_encoded = (
    meals
    .select(pl.col('meal_id').unique())
    .with_row_index('meal_id_enc')
)
meals = meals.join(meals_ids_encoded, on='meal_id')
                   


meals.head()

# Save processed data

In [ ]:
with open('src/embedding_tuning/conf.yaml') as file:
    conf = yaml.safe_load(file)

In [ ]:
meals.write_parquet(conf['PATHS']['meals'])